# Notebook 14 — Scikit-learn Preprocessing Pipeline
### Sprint 5 | Data Cleaning & Preprocessing for AI/ML Engineers


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("telco_churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')   # deliberately NOT filled yet — the pipeline will do it
print(f"Dataset ready: {df.shape[0]:,} rows. Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")


Dataset ready: 7,043 rows. Missing TotalCharges: 11


---
## 1. Why Use Pipelines?

### Understand
A `Pipeline` chains preprocessing steps and a final model into one object, fit and used
together — this isn't just convenience, it's exactly what structurally *enforces* the
"fit on train only" discipline from Notebooks 12-13. Calling `.fit()` on a pipeline with
training data only, and `.transform()`/`.predict()` on test data, makes it far harder to
accidentally leak information the way the manual, step-by-step code in Notebook 13 could.

### Demonstrate
**AI/ML use case:** Every preprocessing decision made across Notebooks 2-11 in this
sprint — type conversion, imputation, encoding, scaling — can be captured in ONE pipeline
object, callable identically on any new batch of customers.


---
## 2. `Pipeline` & 3. `ColumnTransformer`

### Understand
- **`Pipeline`**: a linear sequence of steps applied to ONE set of columns (e.g.,
  impute → scale, for numeric columns).
- **`ColumnTransformer`**: applies DIFFERENT pipelines to DIFFERENT column groups in
  parallel, then combines the results — exactly what's needed here, since numeric and
  categorical columns need entirely different treatment.

### Implement


In [2]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
                         'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                         'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
                         'PaperlessBilling', 'PaymentMethod']
print(f"{len(numeric_features)} numeric features, {len(categorical_features)} categorical features")


4 numeric features, 15 categorical features


---
## 4. Numerical Pipeline (5. Imputation Pipeline + 6. Scaling Pipeline combined)

### Implement


In [3]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),   # matches Notebook 3's justified decision
    ('scaler', StandardScaler())
])
print(numeric_pipeline)


Pipeline(steps=[('imputer', SimpleImputer(fill_value=0, strategy='constant')),
                ('scaler', StandardScaler())])


**Reason for this exact recipe:** `SimpleImputer(strategy='constant', fill_value=0)`
directly encodes Notebook 3's documented decision (MAR missingness, `tenure==0` implies
`TotalCharges=0`) — NOT a generic mean/median default. `StandardScaler` follows,
matching Notebook 8's decision for this outlier-free numeric data.


---
## 7. Categorical Pipeline (Imputation + Encoding Pipeline combined)

### Implement


In [4]:
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),   # mode imputation, for any future missing categoricals
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])
print(categorical_pipeline)


Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OneHotEncoder(drop='first', handle_unknown='ignore'))])


**Reason:** `OneHotEncoder(handle_unknown='ignore')` directly matches Notebook 7's
finding — the scikit-learn encoder (not `pd.get_dummies()`) is the production-safe
choice, since it won't crash on a category never seen during training.


---
## 8. Combining Multiple Transformations — `ColumnTransformer`

### Implement


In [5]:
preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])
print(preprocessor)


ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant')),
                                                 ('scaler', StandardScaler())]),
                                 ['tenure', 'MonthlyCharges', 'TotalCharges',
                                  'SeniorCitizen']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'))]),
                                 ['gender', 'Partner', 'Dependents',
                

---
## 9. `fit()`, 10. `transform()`, 11. `fit_transform()`

### Understand
- **`.fit()`**: learns parameters (mean/std, categories, fill values) from the data —
  used ONLY on training data.
- **`.transform()`**: applies previously-learned parameters to new data — used on
  validation/test data, without re-learning anything.
- **`.fit_transform()`**: does both at once — a convenience shortcut, appropriate ONLY
  for the training data.

### Implement — the Full Pipeline, Correctly Fit


In [6]:
X = df[numeric_features + categorical_features]
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# fit_transform on TRAIN only
X_train_processed = preprocessor.fit_transform(X_train)
# transform ONLY (never fit again) on TEST
X_test_processed = preprocessor.transform(X_test)

print(f"Training set: {X_train.shape} -> {X_train_processed.shape} after preprocessing")
print(f"Test set    : {X_test.shape} -> {X_test_processed.shape} after preprocessing")
print(f"\nAny NaN remaining in processed training data: {np.isnan(X_train_processed).sum()}")


Training set: (5634, 19) -> (5634, 30) after preprocessing
Test set    : (1409, 19) -> (1409, 30) after preprocessing

Any NaN remaining in processed training data: 0


---
## Full Pipeline Including the Model

### Documentation (Problem / Analysis / Technique / Reason / Implementation / Result / Impact)
- **Problem:** Notebooks 2-11's preprocessing decisions were applied manually,
  step-by-step — reproducible in code, but not packaged as a single reusable object.
- **Analysis:** A `Pipeline` + `ColumnTransformer` combination can encode every one of
  those decisions (constant imputation, StandardScaler, OneHotEncoder) in one object.
- **Technique Selected:** A single end-to-end `Pipeline` combining the
  `ColumnTransformer` with a final `LogisticRegression` step.
- **Reason:** Structurally prevents the exact leakage patterns demonstrated in
  Notebook 13 — calling `.fit()` once on training data fits every internal step
  correctly-scoped, automatically.
- **Implementation:** below.
- **Result:** A single `.fit()` / `.predict()` interface for the entire preprocessing +
  modeling workflow.
- **Impact:** This is the form a real production system would actually deploy — not
  loose, manually-sequenced code cells.


In [7]:
full_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

full_pipeline.fit(X_train, y_train)   # ONE call fits every step correctly, train-only
test_accuracy = full_pipeline.score(X_test, y_test)
print(f"Full pipeline test accuracy: {test_accuracy:.4f}")

# Demonstrating that the SAME pipeline object handles brand-new, unseen data cleanly
new_customer = X_test.iloc[[0]]
prediction = full_pipeline.predict(new_customer)
print(f"\nPrediction for one new customer record: {'Churn' if prediction[0]==1 else 'No Churn'}")


Full pipeline test accuracy: 0.7388

Prediction for one new customer record: No Churn


---
## Summary

| Stage | Component | Encodes Which Earlier Decision |
|---|---|---|
| Numeric imputation | `SimpleImputer(strategy='constant', fill_value=0)` | Notebook 3's MAR-justified constant fill |
| Numeric scaling | `StandardScaler` | Notebook 8's outlier-free scaling choice |
| Categorical imputation | `SimpleImputer(strategy='most_frequent')` | Mode imputation (Sprint 2/3 convention) |
| Categorical encoding | `OneHotEncoder(handle_unknown='ignore')` | Notebook 7's production-safe encoding choice |
| Combination | `ColumnTransformer` | Applies numeric/categorical pipelines in parallel |
| Full workflow | `Pipeline` (preprocessing + model) | Enforces fit-on-train discipline (Notebooks 12-13) |

**Next notebook:** `15_Before_After_Preprocessing.ipynb` — a full, side-by-side
comparison of the dataset before this sprint's work and after.
